# Football Match Prediction - ML Prediction Demo

This notebook demonstrates machine learning predictions for match outcomes.

## 1. Load and Prepare Data

In [ ]:
import pandas as pd
import numpy as np
from src.preprocessing.feature_engineer import FeatureEngineer
from src.prediction.ml_predictor import MLPredictor

# Load data
matches_df = pd.read_csv('data/processed/matches.csv')
print(f'Loaded {len(matches_df)} matches')

## 2. Feature Engineering

In [ ]:
# Create features
print('Creating features...')
df_features = FeatureEngineer.feature_engineering_pipeline(matches_df)

print(f'Features created. Shape: {df_features.shape}')
print(f'\nNew columns: {[col for col in df_features.columns if col not in matches_df.columns]}')

## 3. Prepare Training Data

In [ ]:
# Create target variable
if 'home_goals' in df_features.columns and 'away_goals' in df_features.columns:
    df_features['result'] = np.where(
        df_features['home_goals'] > df_features['away_goals'], 1,
        np.where(df_features['home_goals'] == df_features['away_goals'], 0, -1)
    )
    print('Target variable created')
    print(df_features['result'].value_counts())
else:
    print('Warning: Goals columns not found')

In [ ]:
# Extract features and target
numeric_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in ['result']]

X = df_features[numeric_cols].fillna(0).values
y = df_features['result'].values

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

## 4. Train Models

In [ ]:
# Train XGBoost model
print('Training XGBoost model...')
predictor_xgb = MLPredictor(model_type='xgboost')
metrics_xgb = predictor_xgb.train(X, y)

print(f'XGBoost Results:')
for key, value in metrics_xgb.items():
    print(f'  {key}: {value}')

In [ ]:
# Train Random Forest model
print('Training Random Forest model...')
predictor_rf = MLPredictor(model_type='random_forest')
metrics_rf = predictor_rf.train(X, y)

print(f'Random Forest Results:')
for key, value in metrics_rf.items():
    print(f'  {key}: {value}')

In [ ]:
# Train Logistic Regression model
print('Training Logistic Regression model...')
predictor_lr = MLPredictor(model_type='logistic')
metrics_lr = predictor_lr.train(X, y)

print(f'Logistic Regression Results:')
for key, value in metrics_lr.items():
    print(f'  {key}: {value}')

## 5. Model Comparison

In [ ]:
import pandas as pd

# Compare models
comparison = pd.DataFrame({
    'XGBoost': metrics_xgb,
    'Random Forest': metrics_rf,
    'Logistic Regression': metrics_lr
})

print('Model Comparison:')
print(comparison)

## 6. Save Best Model

In [ ]:
import os

os.makedirs('src/prediction/models', exist_ok=True)

# Save XGBoost as best model
predictor_xgb.save_model('src/prediction/models/best_model.pkl')
print('Model saved successfully!')